# Retrieval-Augmented Generation (RAG) con Gemini AI

## 1. Configuración (Setup)

Este cuaderno implementa un sistema básico de Retrieval-Augmented Generation (RAG) utilizando **Google Gemini AI** y embeddings de **HuggingFace**.

**Nota:** Este material ha sido elaborado para la clase de Técnicas de ML en la Pontificia Universidad Javeriana.

### Requisitos previos:
- Un archivo PDF para ser analizado (por defecto `reglamento.pdf`).
- Un token de API de Google Gemini configurado en los secretos de Colab bajo el nombre `GEMINI_API_KEY`.

In [ ]:
# @title Instalación de dependencias
!pip install -q google-genai gradio pypdf "langchain-huggingface[full]" langchain-text-splitters tqdm "numpy<2"

A continuación, importamos las librerías necesarias para el procesamiento de texto, la generación de embeddings, el cálculo de similitudes y la interfaz de usuario.

⚠ Se requiere reiniciar la sesión después de instaladas las dependencias.

In [ ]:
# @title Importación de librerías
import gradio as gr
from google.colab import userdata
from google import genai
from pypdf import PdfReader
from tqdm import tqdm
from sentence_transformers import util
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import torch
import os

## 2. Introducción Teórica

El sistema **RAG** (Retrieval-Augmented Generation) mejora las respuestas de los grandes modelos de lenguaje (LLMs) al recuperar información relevante de una base de conocimiento externa antes de generar una respuesta.

El proceso se divide típicamente en:
1.  **Indexación:** Dividir documentos en fragmentos (chunks) y convertirlos en representaciones vectoriales (embeddings).
2.  **Recuperación (Retrieval):** Dada una consulta del usuario, se convierte a vector y se busca en el índice los fragmentos más similares utilizando la similitud del coseno:

    $$\text{Similitud}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$$
    
    Donde $A$ y $B$ son los vectores (embeddings) de la consulta y del fragmento, respectivamente.
3.  **Generación:** Se pasa la consulta original junto con los fragmentos recuperados al LLM para que genere una respuesta informada.

## 3. Implementación (Paso a Paso)

### 3.1 Configuración del Cliente y Modelos

Inicializamos el cliente de Gemini y configuramos los modelos para la generación de texto y embeddings (usando un modelo preentrenado en español).

In [ ]:
# @title Configuración de Gemini y Embeddings
# Asegúrate de tener GEMINI_API_KEY en los secretos de Colab
try:
    client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    print("Cliente Gemini configurado exitosamente.")
except Exception as e:
    print(f"Error al configurar Gemini: {e}\nPor favor, configura 'GEMINI_API_KEY' en los secretos.")

# Configuración del modelo de Embeddings y procesamiento de texto
# Usamos un modelo optimizado para similitud de oraciones en español
embeddings = HuggingFaceEmbeddings(model_name="hiiamsid/sentence_similarity_spanish_es")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
print("Modelos de embedding cargados.")

### 3.2 Funciones de Procesamiento y Recuperación

Definimos las funciones para extraer texto del PDF, dividirlo en fragmentos, calcular los embeddings y encontrar los fragmentos más relevantes para una consulta.

In [ ]:
# @title Definición de funciones RAG
def extraer_texto_pdf(ruta_pdf):
    """Extrae el texto de un PDF y lo concatena en un solo string."""
    texto = ""
    with open(ruta_pdf, 'rb') as archivo_pdf:
        lector_pdf = PdfReader(archivo_pdf)
        for pagina in tqdm(lector_pdf.pages, desc="Extrayendo texto"):
            texto += pagina.extract_text() or ""
    return texto

def preprocesar_texto(texto):
    """Divide el texto en fragmentos y calcula sus embeddings."""
    oraciones = text_splitter.split_text(texto)
    print(f"Procesando {len(oraciones)} fragmentos...")
    oraciones_emb = [embeddings.embed_query(ora) for ora in tqdm(oraciones, desc="Calculando embeddings")]
    return oraciones, torch.tensor(oraciones_emb)

def obtener_fragmentos_relevantes(pregunta, oraciones, oraciones_emb, n_resultados=5):
    """Encuentra los fragmentos más relevantes en base a la similitud de coseno."""
    pregunta_emb = torch.tensor(embeddings.embed_query(pregunta))
    similitudes = util.pytorch_cos_sim(pregunta_emb, oraciones_emb)[0]
    indices = torch.topk(similitudes, n_resultados).indices
    fragmentos = [oraciones[i] for i in indices]
    return "\n".join(fragmentos)

## 4. Visualización e Interactividad

### 4.1 Preparación de los datos

Procesamos el documento PDF para tener la base de conocimiento lista. Puedes usar el panel lateral izquierdo para subir tu archivo.

In [ ]:
# @title Carga y Procesamiento del Documento
from google.colab import files
import os

print("Por favor, sube el archivo PDF a analizar:")
uploaded = files.upload()

oraciones = []
oraciones_emb = None

if uploaded:
    nombre_archivo_pdf = list(uploaded.keys())[0]
    print(f"Procesando el archivo: {nombre_archivo_pdf}")
    texto_pdf = extraer_texto_pdf(nombre_archivo_pdf)
    oraciones, oraciones_emb = preprocesar_texto(texto_pdf)
    print("\n¡Procesamiento completado! El sistema está listo.")
else:
    print("❌ Error: No se subió ningún archivo.")

### 4.2 Interfaz de Chat

Lanzamos la interfaz interactiva usando Gradio. Aquí puedes seleccionar qué versión del modelo Gemini usar para generar las respuestas.

In [ ]:
# @title Ejecución del Asistente RAG (Actualizado Abril 2026)
# @markdown Selecciona el modelo de Gemini más reciente:
# Gemini 3.1 Pro es ideal para razonamiento complejo en RAG.
# Gemini 3.1 Flash ofrece un equilibrio perfecto entre velocidad y capacidad.
modelo_gemini = "gemini-3-flash-preview" # @param ["gemini-3-flash-preview", "gemini-3-flash-lite-preview", "gemini-3-pro-preview"]

def gemini_rag_chat(prompt, history=[]):
    """Genera una respuesta usando Gemini basada en el contexto del PDF."""
    if oraciones_emb is None:
        return "Error: Primero debes procesar un documento PDF en la celda anterior."

    # Recuperar fragmentos relevantes
    contexto = obtener_fragmentos_relevantes(prompt, oraciones, oraciones_emb)

    # Formatear historial para el prompt
    hist_text = "\n".join([f"User: {u}\nAI: {a}" for u, a in history])

    # Construir prompt con contexto
    # Optimizamos el system_instruction para los modelos Gemini 3.1
    prompt_con_contexto = f"Contexto del documento:\n{contexto}\n\nPregunta: {prompt}"

    try:
        # Llamada a Gemini utilizando la estructura de la SDK más reciente
        response = client.models.generate_content(
            model=modelo_gemini,
            contents=f"{hist_text}\nUser: {prompt_con_contexto}",
            config=genai.types.GenerateContentConfig(
                system_instruction="Eres un asistente experto en análisis de documentos. Tu tarea es responder consultas basándote exclusivamente en el contexto proporcionado, manteniendo un tono académico y preciso."
            )
        )
        return response.text
    except Exception as e:
         return f"Ocurrió un error al contactar a Gemini ({modelo_gemini}): {e}"

# Lanzar interfaz
if oraciones_emb is not None:
    print(f"Iniciando interfaz con el modelo de última generación: {modelo_gemini}")
    gr.ChatInterface(
        fn=gemini_rag_chat,
        title="Asistente RAG Avanzado - Gemini 3.1",
        description=f"Consultas técnicas sobre el documento usando {modelo_gemini}."
    ).launch(debug=True, share=True)
else:
    print("Error: No se detectaron embeddings. Procesa el PDF antes de iniciar la interfaz.")